In [0]:
%run ./00_Setup

In [0]:
# ============================================
# IMPORT LIBRARIES
# ============================================

import json
import random
import time
import uuid

from datetime import datetime
from pyspark.sql import Row

In [0]:
# ============================================
# GENERATOR CONFIGURATION
# ============================================

INTERVAL_SECONDS = 10

print(f"Data generation interval: {INTERVAL_SECONDS} seconds")

print("Customers path:", CUSTOMERS_PATH)
print("Books path    :", BOOKS_PATH)
print("Orders path   :", ORDERS_PATH)

In [0]:
# ============================================
# CUSTOMER DATA GENERATOR
# ============================================

def generate_customer():

    customer_id = str(uuid.uuid4())

    first_names = [
        "Arun",
        "Priya",
        "Rahul",
        "Sneha",
        "Vijay",
        "Anita",
        "Karthik",
        "Divya"
    ]

    last_names = [
        "Kumar",
        "Sharma",
        "Reddy",
        "Patel",
        "Iyer",
        "Singh",
        "Das",
        "Nair"
    ]

    first_name = random.choice(first_names)
    last_name = random.choice(last_names)

    return {
        "customer_id": customer_id,
        "first_name": first_name,
        "last_name": last_name,
        "email": f"{first_name.lower()}.{last_name.lower()}@example.com",
        "city": random.choice([
            "Chennai",
            "Bangalore",
            "Hyderabad",
            "Mumbai",
            "Delhi"
        ]),
        "created_at": datetime.now().isoformat()
    }

In [0]:
# ============================================
# BOOK DATA GENERATOR
# ============================================

def generate_book():

    book_id = str(uuid.uuid4())

    books = [
        ("Learning Python", "Programming", 450),
        ("Spark Fundamentals", "Big Data", 650),
        ("Data Engineering with Databricks", "Data Engineering", 800),
        ("SQL for Beginners", "Database", 400),
        ("Azure Data Engineering", "Cloud", 700),
        ("Machine Learning Basics", "Machine Learning", 750),
        ("Advanced Spark", "Big Data", 900)
    ]

    title, category, price = random.choice(books)

    return {
        "book_id": book_id,
        "title": title,
        "category": category,
        "price": price,
        "created_at": datetime.now().isoformat()
    }

In [0]:
# ============================================
# ORDER DATA GENERATOR
# ============================================

def generate_order(customer_id, book_id, price):

    quantity = random.randint(1, 3)

    return {
        "order_id": str(uuid.uuid4()),
        "customer_id": customer_id,
        "book_id": book_id,
        "quantity": quantity,
        "unit_price": price,
        "total_amount": quantity * price,
        "order_status": random.choice([
            "PLACED",
            "CONFIRMED",
            "SHIPPED"
        ]),
        "order_timestamp": datetime.now().isoformat()
    }

In [0]:
# ============================================
# WRITE CUSTOMER CSV
# ============================================

def write_customer_csv():

    customer = generate_customer()

    file_name = (
        f"{CUSTOMERS_PATH}/"
        f"customer_{datetime.now().strftime('%Y%m%d_%H%M%S_%f')}.csv"
    )

    customer_df = spark.createDataFrame([Row(**customer)])

    customer_df.coalesce(1) \
        .write \
        .mode("overwrite") \
        .option("header", "true") \
        .csv(file_name)

    print(f"Customer CSV created: {file_name}")

    return customer

In [0]:
# ============================================
# WRITE BOOK JSON
# ============================================

def write_book_json():

    book = generate_book()

    file_name = (
        f"{BOOKS_PATH}/"
        f"book_{datetime.now().strftime('%Y%m%d_%H%M%S_%f')}.json"
    )

    book_df = spark.createDataFrame([Row(**book)])

    book_df.coalesce(1) \
        .write \
        .mode("overwrite") \
        .json(file_name)

    print(f"Book JSON created: {file_name}")

    return book

In [0]:
# ============================================
# WRITE ORDER PARQUET
# ============================================

def write_order_parquet(customer, book):

    order = generate_order(
        customer["customer_id"],
        book["book_id"],
        book["price"]
    )

    file_name = (
        f"{ORDERS_PATH}/"
        f"order_{datetime.now().strftime('%Y%m%d_%H%M%S_%f')}.parquet"
    )

    order_df = spark.createDataFrame([Row(**order)])

    order_df.coalesce(1) \
        .write \
        .mode("overwrite") \
        .parquet(file_name)

    print(f"Order Parquet created: {file_name}")

    return order

In [0]:
# ============================================
# GENERATE ONE BATCH
# ============================================

customer = write_customer_csv()

book = write_book_json()

order = write_order_parquet(customer, book)

print("\nBatch generated successfully.")
print("Customer:", customer)
print("Book    :", book)
print("Order   :", order)

In [0]:
# ============================================
# CHECK GENERATED FILES
# ============================================

print("CUSTOMERS")
display(dbutils.fs.ls(CUSTOMERS_PATH))

print("BOOKS")
display(dbutils.fs.ls(BOOKS_PATH))

print("ORDERS")
display(dbutils.fs.ls(ORDERS_PATH))

In [0]:
# ============================================
# CONTINUOUS DATA GENERATOR
# ============================================

print("Starting continuous data generation...")
print("Press the Stop button to stop the generator.")

while True:

    try:

        # Generate customer
        customer = write_customer_csv()

        # Generate book
        book = write_book_json()

        # Generate order
        order = write_order_parquet(customer, book)

        print(
            f"\nBatch completed at "
            f"{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}"
        )

        print(f"Waiting {INTERVAL_SECONDS} seconds...\n")

        time.sleep(INTERVAL_SECONDS)

    except Exception as e:

        print("Generator stopped because of an error:")
        print(e)
        break